### SCD Type 2 on Customer Table

In [0]:
from pyspark.sql.functions import *

#  Source
df = spark.read.format("delta").table("bankingdata.silver.account")

In [0]:
df.columns


### Hash Column at Source


In [0]:
df_hash = df.withColumn(
    "src_hash",
    crc32(concat_ws("||",
        col("AccountID").cast("string"),
        col("CustomerID").cast("string"),
        col("BranchID").cast("string"),
        col("AccountType"),
        col("Balance").cast("string"),
        col("Status")
    ))
)

### Target Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bankingdata.gold.dim_account
(
    AccountID INT,
    CustomerID INT,
    BranchID INT,
    AccountType STRING,
    Balance DOUBLE,
    Status STRING,
    ModifiedDate TIMESTAMP,

    HASHVALUE STRING,

    CREATEDDATE TIMESTAMP,
    CREATEDBY STRING,
    UPDATEDDATE TIMESTAMP,
    UPDATEDBY STRING,

    IS_ACTIVE INT
)

In [0]:
# Target table
from delta.tables import DeltaTable
table_name = "bankingdata.gold.dim_account"
df_tgt = DeltaTable.forName(spark, table_name)

### Detect New or Changed Records

In [0]:
df_new = (
    df_hash.alias("src")
    .join(
        df_tgt.toDF().alias("tgt"),
        (col("src.AccountID") == col("tgt.AccountID")) &
        (col("src.src_hash") == col("tgt.HASHVALUE")),
        "left_anti"
    )
)

### Identify Changed Records

In [0]:
LatestRecord = (
    df_new.alias("new")
    .join(
        df_tgt.toDF().alias("old"),
        col("new.AccountID") == col("old.AccountID"),
        "inner"
    )
    .where(
        (col("new.src_hash") != col("old.HASHVALUE")) &
        (col("old.IS_ACTIVE") == 1)
    )
    .select(
        col("new.AccountID").alias("MergeKey"),
        col("new.*")
    )
)


### Prepare New Records


In [0]:
from pyspark.sql.functions import lit

LatestRecord1 = (
    df_new
    .select(
        lit(None).alias("MergeKey"),
        "*"
    )
)

### Combine Changed and New Records

In [0]:
updates = LatestRecord.union(LatestRecord1)


### Apply SCD Type-2 Using Delta MERGE


In [0]:
(
    df_tgt.alias("tgt")
    .merge(
        updates.alias("src"),
        "tgt.AccountID = src.MergeKey AND tgt.IS_ACTIVE = 1"
    )
    
    # 🔹 Expire old record if data changed
    .whenMatchedUpdate(
        condition="tgt.IS_ACTIVE = 1 AND tgt.HASHVALUE != src.src_hash",
        set={
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("Databricks-Updated"),
            "IS_ACTIVE": lit(0)
        }
    )
    
    # 🔹 Insert new record (new OR changed)
    .whenNotMatchedInsert(
        values={
            "AccountID": "src.AccountID",
            "CustomerID": "src.CustomerID",
            "BranchID": "src.BranchID",
            "AccountType": "src.AccountType",
            "Balance": "src.Balance",
            "Status": "src.Status",
            "ModifiedDate": "src.ModifiedDate",
            "HASHVALUE": "src.src_hash",
            "CREATEDDATE": current_timestamp(),
            "CREATEDBY": lit("databricks"),
            "UPDATEDDATE": to_timestamp(lit("9999-01-01 00:00:00")),
            "UPDATEDBY": lit("databricks"),
            "IS_ACTIVE": lit(1)
        }
    )
    
    .execute()
)

In [0]:
%sql
select * from bankingdata.gold.dim_account